# SalesLuv 딜 승패 모델 2차 튜닝과 앙상블

이 노트북은 1차 결과를 기준점으로 삼아 모델별 탐색 범위를 좁혀 다시 튜닝하고,
확률 보정·Soft Voting·Stacking을 같은 조건에서 비교합니다.

- 입력: 22개 범주형 컬럼
- 정답: `Lost=0`, `Won=1`
- 완전히 같은 23개 값의 중복 행 제거: 448건 → 365건
- 원본의 `Unknown`은 범주로 유지하고 합성 `Unknown`은 만들지 않음
- 분할: Train 70% / Test 30%, `random_state=1`, 층화 추출
- 하이퍼파라미터 기준: Train 내부 5-Fold CV Brier Score 단일 지표
- 튜닝 결과: 후보별 평균 CV 점수, 최적 파라미터, 최적 CV 점수 확인
- 최종 비교: CV Brier와 Test Brier·AUC·Accuracy 비교
- 임계값 조정과 FP·FN 비용 계산: 3차 노트북에서 진행
- 모델 저장: 3차 결과를 확인하고 최종 구조를 확정한 뒤 진행


## 0. 환경과 경로

저장소 루트에서 다음 명령으로 전용 환경의 JupyterLab을 실행합니다.

```bash
uv run --project backend/notebooks --locked jupyter lab \
  backend/notebooks/deal_model_phase2.ipynb
```

원본 CSV가 기본 경로(`/private/tmp/Salvirt_B2B_ML_dataset_HF.csv`)와 다르면 `SALESLUV_B2B_DATA_PATH` 환경변수로 지정합니다.

In [1]:
from __future__ import annotations

import hashlib
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tabicl import TabICLClassifier

SOURCE_SHA256 = "8dee635b95bdcb00896b654efe62fc20177090081c81ef5224e8641ba31c3061"
RANDOM_STATE = 1

ALL_COLUMNS = (
    "Product",
    "Seller",
    "Authority",
    "Comp_size",
    "Competitors",
    "Purch_dept",
    "Partnership",
    "Budgt_alloc",
    "Forml_tend",
    "RFI",
    "RFP",
    "Growth",
    "Posit_statm",
    "Source",
    "Client",
    "Scope",
    "Strat_deal",
    "Cross_sale",
    "Up_sale",
    "Deal_type",
    "Needs_def",
    "Att_t_client",
    "Status",
)
FEATURE_NAMES = ALL_COLUMNS[:-1]
DATA_PATH = Path(
    os.environ.get(
        "SALESLUV_B2B_DATA_PATH",
        "/private/tmp/Salvirt_B2B_ML_dataset_HF.csv",
    )
).expanduser()

print(f"데이터: {DATA_PATH}")

데이터: /private/tmp/Salvirt_B2B_ML_dataset_HF.csv


## 1. 데이터 준비

In [2]:
# 1차와 다른 CSV를 사용하면 점수를 직접 비교할 수 없으므로 원본 해시를 확인합니다.
source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert source_sha256 == SOURCE_SHA256, f"unexpected source: {source_sha256}"

# 원본은 세미콜론으로 컬럼을 구분합니다. 문자열의 앞뒤 공백을 정리한 뒤 완전 중복만 제거합니다.
data = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig", dtype=str)
data = data.apply(lambda column: column.str.strip())
assert tuple(data.columns) == ALL_COLUMNS
assert not data.isna().any().any()
assert not data.eq("").any().any()

raw_rows = len(data)
data = data.drop_duplicates().reset_index(drop=True)
deduplicated_rows = len(data)

assert raw_rows == 448
assert deduplicated_rows == 365
assert data["Status"].value_counts().to_dict() == {"Lost": 192, "Won": 173}

print(f"원본 행: {raw_rows}")
print(f"중복 제거 후: {deduplicated_rows}")
print(f"제거된 중복: {raw_rows - deduplicated_rows}")
display(data.head(10))

원본 행: 448
중복 제거 후: 365
제거된 중복: 83


In [3]:
target = "Status"
X = data.drop(columns=target).astype(str)
y = data[target].map({"Lost": 0, "Won": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.7,
    random_state=RANDOM_STATE,
    stratify=y,
)

# cv=5가 만드는 것과 같은 StratifiedKFold를 목록으로 고정해 모든 2차 후보가 같은 행을 평가합니다.
cv5 = list(StratifiedKFold(n_splits=5, shuffle=False).split(X_train, y_train))

print(f"Train: {len(X_train)}, {dict(sorted(Counter(y_train).items()))}")
print(f"Test: {len(X_test)}, {dict(sorted(Counter(y_test).items()))}")

Train: 255, {0: 134, 1: 121}
Test: 110, {0: 58, 1: 52}


In [4]:
def won_probabilities(model, X_data) -> np.ndarray:
    """모델의 클래스 순서를 확인해 Won=1 확률 열만 반환합니다."""
    won_index = list(model.classes_).index(1)
    return model.predict_proba(X_data)[:, won_index]


def calculate_test_metrics(model) -> dict[str, float]:
    """학습된 모델의 Brier, AUC, Accuracy를 Test에서 계산합니다."""
    probabilities = won_probabilities(model, X_test)
    predictions = model.predict(X_test)
    return {
        "brier": float(brier_score_loss(y_test, probabilities)),
        "auc": float(roc_auc_score(y_test, probabilities)),
        "accuracy": float(accuracy_score(y_test, predictions)),
    }


def best_search_fold_briers(search: GridSearchCV) -> np.ndarray:
    """SearchCV가 선택한 후보의 5개 Fold Brier를 같은 순서로 반환합니다."""
    return np.asarray(
        [
            -search.cv_results_[f"split{fold}_test_score"][search.best_index_]
            for fold in range(len(cv5))
        ],
        dtype=float,
    )


def print_search_result(label: str, baseline_scores: np.ndarray, search: GridSearchCV) -> None:
    """기본 CV와 후보별 탐색 결과를 일정한 형식으로 출력합니다."""
    print(f"[{label}]")
    print(f"기본 CV Brier: {-baseline_scores.mean():.6f}")
    print("후보별 CV Brier:", -search.cv_results_["mean_test_score"])
    print(f"최적 파라미터: {search.best_params_}")
    print(f"최적 CV 점수: {search.best_score_:.6f}")
    print(f"최적 CV Brier: {-search.best_score_:.6f}")

## 2. 1차 결과 기준점

아래 값은 같은 데이터·분할로 실행한 1차 노트북의 결과입니다. 2차 모델 선택 계산에는
사용하지 않고, 새 탐색 결과가 실제로 개선됐는지 확인하는 기준점으로만 표시합니다.

In [5]:
phase1_comparison = pd.DataFrame(
    [
        {
            "model": "LogisticRegression",
            "cv_brier": 0.161561,
            "test_brier": 0.169427,
            "test_auc": 0.818800,
            "test_accuracy": 0.754545,
            "best_params": {"classifier__C": 0.1},
        },
        {
            "model": "TabICL",
            "cv_brier": 0.163147,
            "test_brier": 0.163944,
            "test_auc": 0.837367,
            "test_accuracy": 0.772727,
            "best_params": {
                "norm_methods": "quantile",
                "n_estimators": 8,
                "softmax_temperature": 0.9,
            },
        },
        {
            "model": "MultinomialNB",
            "cv_brier": 0.164812,
            "test_brier": 0.177516,
            "test_auc": 0.821121,
            "test_accuracy": 0.763636,
            "best_params": {"estimator__classifier__alpha": 0.3},
        },
        {
            "model": "ExtraTrees",
            "cv_brier": 0.168015,
            "test_brier": 0.170509,
            "test_auc": 0.826426,
            "test_accuracy": 0.800000,
            "best_params": {
                "classifier__max_depth": 8,
                "classifier__max_features": "sqrt",
                "classifier__min_samples_leaf": 4,
                "classifier__n_estimators": 300,
            },
        },
        {
            "model": "CatBoost",
            "cv_brier": 0.172698,
            "test_brier": 0.167629,
            "test_auc": 0.829410,
            "test_accuracy": 0.763636,
            "best_params": {
                "depth": 4,
                "iterations": 500,
                "learning_rate": 0.025,
                "l2_leaf_reg": 10.0,
                "random_strength": 2.0,
            },
        },
    ]
).sort_values("cv_brier", ignore_index=True)

display(phase1_comparison)

,model,cv_brier,test_brier,test_auc,test_accuracy,best_params
0,LogisticRegression,0.161561,0.169427,0.818800,0.754545,{'classifier__C': 0.1}
1,TabICL,0.163147,0.163944,0.837367,0.772727,"{'norm_methods': 'quantile', 'n_estimators': 8..."
2,MultinomialNB,0.164812,0.177516,0.821121,0.763636,{'estimator__classifier__alpha': 0.3}
3,ExtraTrees,0.168015,0.170509,0.826426,0.800000,"{'classifier__max_depth': 8, 'classifier__max_..."
4,CatBoost,0.172698,0.167629,0.829410,0.763636,"{'depth': 4, 'iterations': 500, 'learning_rate..."


## 3. LogisticRegression 2차 튜닝

1차 최적값 `C=0.1` 주변을 더 촘촘하게 탐색합니다.

In [6]:
model_logistic_phase1 = Pipeline(
    [
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        (
            "classifier",
            LogisticRegression(
                C=0.1,
                l1_ratio=0,
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

cv_scores_logistic_phase1 = cross_val_score(
    model_logistic_phase1,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
)

param_logistic_phase2 = {
    "classifier__C": [0.03, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3],
}
search_logistic_phase2 = GridSearchCV(
    model_logistic_phase1,
    param_logistic_phase2,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_logistic_phase2.fit(X_train, y_train)

print_search_result("LogisticRegression", cv_scores_logistic_phase1, search_logistic_phase2)

[LogisticRegression]
기본 CV Brier: 0.161561
후보별 CV Brier: [0.16830069 0.16374591 0.16198622 0.16156133 0.16189057 0.16266175
 0.16439495]
최적 파라미터: {'classifier__C': 0.1}
최적 CV 점수: -0.161561
최적 CV Brier: 0.161561


## 4. MultinomialNB 2차 튜닝

1차 탐색의 최솟값이 하한 `alpha=0.3`이었으므로 더 작은 alpha와 보정기 구성까지 비교합니다.

In [7]:
model_nb_phase1 = CalibratedClassifierCV(
    estimator=Pipeline(
        [
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ("classifier", MultinomialNB(alpha=0.3, fit_prior=True)),
        ]
    ),
    method="sigmoid",
    cv=5,
    ensemble="auto",
)

cv_scores_nb_phase1 = cross_val_score(
    model_nb_phase1,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
)

param_nb_phase2 = {
    "estimator__classifier__alpha": [0.03, 0.05, 0.1, 0.2, 0.3, 0.45, 0.6],
    "ensemble": ["auto", False],
}
search_nb_phase2 = GridSearchCV(
    model_nb_phase1,
    param_nb_phase2,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_nb_phase2.fit(X_train, y_train)

print_search_result("MultinomialNB", cv_scores_nb_phase1, search_nb_phase2)

[MultinomialNB]
기본 CV Brier: 0.164812
후보별 CV Brier: [0.1642375  0.16388625 0.16386971 0.16436296 0.16481156 0.16528893
 0.16559785 0.16243601 0.16223407 0.16256997 0.16347451 0.16414894
 0.16483458 0.16528375]
최적 파라미터: {'ensemble': False, 'estimator__classifier__alpha': 0.05}
최적 CV 점수: -0.162234
최적 CV Brier: 0.162234


## 5. ExtraTrees 2차 튜닝

1차 최적 조합을 반드시 포함하고, 깊이·잎 크기·입력 비율·분할 기준을 중심으로 73개 조합을
비교합니다. 트리 내부 병렬화는 끄고 Grid Search에서 후보를 병렬 실행합니다.

In [8]:
model_extratrees_phase1 = Pipeline(
    [
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        (
            "classifier",
            ExtraTreesClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=4,
                max_features="sqrt",
                criterion="gini",
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)

cv_scores_extratrees_phase1 = cross_val_score(
    model_extratrees_phase1,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
)

param_extratrees_phase2 = [
    {
        "classifier__n_estimators": [300],
        "classifier__max_depth": [8],
        "classifier__min_samples_leaf": [4],
        "classifier__max_features": ["sqrt"],
        "classifier__criterion": ["gini"],
    },
    {
        "classifier__n_estimators": [600],
        "classifier__max_depth": [6, 8, 10],
        "classifier__min_samples_leaf": [3, 4, 6, 8],
        "classifier__max_features": ["sqrt", 0.15, 0.25],
        "classifier__criterion": ["gini", "log_loss"],
    },
]
search_extratrees_phase2 = GridSearchCV(
    model_extratrees_phase1,
    param_extratrees_phase2,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_extratrees_phase2.fit(X_train, y_train)

print_search_result("ExtraTrees", cv_scores_extratrees_phase1, search_extratrees_phase2)

[ExtraTrees]
기본 CV Brier: 0.168015
후보별 CV Brier: [0.16801535 0.16790634 0.16776966 0.16703409 0.16728319 0.16955129
 0.1693019  0.16824421 0.168343   0.17615204 0.17448719 0.17370728
 0.17258984 0.16968755 0.16849089 0.16651898 0.16728292 0.17256878
 0.17082888 0.16848653 0.16788335 0.17920554 0.17526294 0.17469762
 0.17301025 0.17083702 0.16865121 0.16660857 0.16733112 0.17324645
 0.17029234 0.16837575 0.16791926 0.17999791 0.17528114 0.17451756
 0.17299524 0.16820981 0.16779137 0.16725304 0.16719485 0.16975573
 0.16953698 0.16884979 0.16822996 0.17662756 0.17511217 0.17362567
 0.17320158 0.17012784 0.16856444 0.16668799 0.16740321 0.17313696
 0.17124254 0.16907794 0.16780322 0.17997257 0.17659573 0.17446648
 0.17304205 0.17105633 0.16863812 0.16666198 0.16744889 0.17405295
 0.17099654 0.16895501 0.16787667 0.18105173 0.17677937 0.17460199
 0.17303375]
최적 파라미터: {'classifier__criterion': 'gini', 'classifier__max_depth': 8, 'classifier__max_features': 'sqrt', 'classifier__min_samples_le

## 6. CatBoost 2차 튜닝

1차 최적 조합에서 한 요소씩 바꾼 9개 구조에 class weight 적용 여부를 더해 18개 후보를
비교합니다. `cat_features`는 생성자에 tuple로
보존해 이후 Voting과 Stacking이 모델을 복제해도 범주형 컬럼 정보가 유지되게 합니다.

In [9]:
model_catboost_phase1 = CatBoostClassifier(
    iterations=500,
    learning_rate=0.025,
    depth=4,
    l2_leaf_reg=10.0,
    random_strength=2.0,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    bootstrap_type="MVS",
    subsample=0.8,
    one_hot_max_size=2,
    cat_features=tuple(FEATURE_NAMES),
    verbose=False,
    allow_writing_files=False,
    random_seed=RANDOM_STATE,
    thread_count=1,
)

cv_scores_catboost_phase1 = cross_val_score(
    model_catboost_phase1,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
)

# CatBoost는 GridSearchCV가 auto_class_weights=None을 set_params로 전달하면 실패합니다.
# 가중치 없는 모델은 해당 파라미터를 생성자에서 완전히 제외해 탐색 기준으로 사용합니다.
model_catboost_phase2 = CatBoostClassifier(
    **{
        name: value
        for name, value in model_catboost_phase1.get_params().items()
        if name != "auto_class_weights"
    }
)

catboost_configs = [
    (4, 500, 0.025, 10.0, 2.0),
    (3, 500, 0.025, 10.0, 2.0),
    (5, 500, 0.025, 10.0, 2.0),
    (4, 350, 0.035, 10.0, 2.0),
    (4, 700, 0.018, 10.0, 2.0),
    (4, 500, 0.025, 5.0, 2.0),
    (4, 500, 0.025, 20.0, 2.0),
    (4, 500, 0.025, 10.0, 1.0),
    (4, 500, 0.025, 10.0, 3.0),
]
param_catboost_phase2 = []
for depth, iterations, learning_rate, l2_leaf_reg, random_strength in catboost_configs:
    config = {
        "depth": [depth],
        "iterations": [iterations],
        "learning_rate": [learning_rate],
        "l2_leaf_reg": [l2_leaf_reg],
        "random_strength": [random_strength],
    }
    param_catboost_phase2.extend([config, {**config, "auto_class_weights": ["Balanced"]}])
search_catboost_phase2 = GridSearchCV(
    model_catboost_phase2,
    param_catboost_phase2,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_catboost_phase2.fit(X_train, y_train)

print_search_result("CatBoost", cv_scores_catboost_phase1, search_catboost_phase2)

[CatBoost]
기본 CV Brier: 0.172698
후보별 CV Brier: [0.17582862 0.17269827 0.17405869 0.1730839  0.1745498  0.1752336
 0.17254416 0.17350709 0.17130485 0.17449202 0.17272324 0.17587928
 0.1718467  0.17069904 0.17495588 0.17340393 0.17357379 0.17209216]
최적 파라미터: {'auto_class_weights': 'Balanced', 'depth': 4, 'iterations': 500, 'l2_leaf_reg': 20.0, 'learning_rate': 0.025, 'random_strength': 2.0}
최적 CV 점수: -0.170699
최적 CV Brier: 0.170699


## 7. TabICL 2차 튜닝

1차 최적값인 `quantile / n_estimators=8 / temperature=0.9`를 포함해 온도와 estimator 수만
좁게 비교합니다. 동일 장치를 동시에 사용하지 않도록 순차 실행합니다.

In [10]:
tabicl_device = "mps" if torch.backends.mps.is_available() else None
model_tabicl_phase1 = TabICLClassifier(
    n_estimators=8,
    batch_size=8,
    kv_cache=False,
    allow_auto_download=True,
    device=tabicl_device,
    use_fa3="auto",
    offload_mode="auto",
    norm_methods="quantile",
    feat_shuffle_method="latin",
    average_logits=True,
    softmax_temperature=0.9,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=False,
)

cv_scores_tabicl_phase1 = cross_val_score(
    model_tabicl_phase1,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
)

param_tabicl_phase2 = [
    {
        "norm_methods": ["quantile"],
        "n_estimators": [8],
        "feat_shuffle_method": ["latin"],
        "average_logits": [True],
        "softmax_temperature": [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3],
    },
    {
        "norm_methods": ["quantile"],
        "n_estimators": [16],
        "feat_shuffle_method": ["latin"],
        "average_logits": [True],
        "softmax_temperature": [0.9, 1.1],
    },
]
search_tabicl_phase2 = GridSearchCV(
    model_tabicl_phase1,
    param_tabicl_phase2,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
    error_score="raise",
)
search_tabicl_phase2.fit(X_train, y_train)

print_search_result("TabICL", cv_scores_tabicl_phase1, search_tabicl_phase2)

[TabICL]
기본 CV Brier: 0.163147
후보별 CV Brier: [0.16673821 0.1643306  0.16314738 0.16290784 0.16337404 0.16435381
 0.1656969  0.16256935 0.16279705]
최적 파라미터: {'average_logits': True, 'feat_shuffle_method': 'latin', 'n_estimators': 16, 'norm_methods': 'quantile', 'softmax_temperature': 0.9}
최적 CV 점수: -0.162569
최적 CV Brier: 0.162569


## 8. 2차 개별 모델 비교

각 SearchCV는 Brier가 가장 낮은 파라미터를 선택합니다. 최적 파라미터로 학습된 모델의
CV Brier와 Test Brier·AUC·Accuracy를 같은 표에서 비교합니다.


In [11]:
phase2_individual_searches = {
    "LogisticRegression_2nd": search_logistic_phase2,
    "MultinomialNB_2nd": search_nb_phase2,
    "ExtraTrees_2nd": search_extratrees_phase2,
    "CatBoost_2nd": search_catboost_phase2,
    "TabICL_2nd": search_tabicl_phase2,
}
phase2_individual_models = {
    name: search.best_estimator_ for name, search in phase2_individual_searches.items()
}
phase2_individual_fold_briers = {
    name: best_search_fold_briers(search) for name, search in phase2_individual_searches.items()
}
phase2_individual_test_metrics = {
    name: calculate_test_metrics(model) for name, model in phase2_individual_models.items()
}
phase2_individual_comparison = pd.DataFrame(
    [
        {
            "model": name,
            "cv_brier": float(phase2_individual_fold_briers[name].mean()),
            **{
                f"test_{metric}": value
                for metric, value in phase2_individual_test_metrics[name].items()
            },
            "best_params": search.best_params_,
        }
        for name, search in phase2_individual_searches.items()
    ]
).sort_values("cv_brier", ignore_index=True)

display(phase2_individual_comparison)

,model,cv_brier,test_brier,test_auc,test_accuracy,best_params
0,LogisticRegression_2nd,0.161561,0.169427,0.818800,0.754545,{'classifier__C': 0.1}
1,MultinomialNB_2nd,0.162234,0.177860,0.825763,0.763636,"{'ensemble': False, 'estimator__classifier__al..."
2,TabICL_2nd,0.162569,0.164790,0.837367,0.790909,"{'average_logits': True, 'feat_shuffle_method'..."
3,ExtraTrees_2nd,0.166519,0.169055,0.830073,0.790909,"{'classifier__criterion': 'gini', 'classifier_..."
4,CatBoost_2nd,0.170699,0.164446,0.835710,0.754545,"{'auto_class_weights': 'Balanced', 'depth': 4,..."


## 9. ExtraTrees 확률 보정

2차 ExtraTrees를 그대로 쓴 경우와 sigmoid 보정기의 `ensemble=False/True`를 비교합니다.
Isotonic은 현재 Train 255건에서 과적합 위험이 커 제외합니다.

In [12]:
model_extratrees_sigmoid_small = CalibratedClassifierCV(
    estimator=clone(search_extratrees_phase2.best_estimator_),
    method="sigmoid",
    cv=5,
    ensemble=False,
    n_jobs=1,
)
model_extratrees_sigmoid_ensemble = CalibratedClassifierCV(
    estimator=clone(search_extratrees_phase2.best_estimator_),
    method="sigmoid",
    cv=5,
    ensemble=True,
    n_jobs=1,
)

cv_scores_extratrees_sigmoid_small = cross_val_score(
    model_extratrees_sigmoid_small,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
)
cv_scores_extratrees_sigmoid_ensemble = cross_val_score(
    model_extratrees_sigmoid_ensemble,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
)

model_extratrees_sigmoid_small.fit(X_train, y_train)
model_extratrees_sigmoid_ensemble.fit(X_train, y_train)

extratrees_probability_models = {
    "ExtraTrees_2nd": search_extratrees_phase2.best_estimator_,
    "ExtraTrees_sigmoid_small": model_extratrees_sigmoid_small,
    "ExtraTrees_sigmoid_ensemble": model_extratrees_sigmoid_ensemble,
}
extratrees_probability_fold_briers = {
    "ExtraTrees_2nd": phase2_individual_fold_briers["ExtraTrees_2nd"],
    "ExtraTrees_sigmoid_small": -cv_scores_extratrees_sigmoid_small,
    "ExtraTrees_sigmoid_ensemble": -cv_scores_extratrees_sigmoid_ensemble,
}
extratrees_probability_test_metrics = {
    name: calculate_test_metrics(model) for name, model in extratrees_probability_models.items()
}
extratrees_calibration_comparison = pd.DataFrame(
    [
        {
            "model": name,
            "cv_brier": float(extratrees_probability_fold_briers[name].mean()),
            "cv_brier_std": float(extratrees_probability_fold_briers[name].std(ddof=1)),
            **{
                f"test_{metric}": value
                for metric, value in extratrees_probability_test_metrics[name].items()
            },
        }
        for name in extratrees_probability_models
    ]
).sort_values("cv_brier", ignore_index=True)

display(extratrees_calibration_comparison)

extratrees_probability_name = extratrees_calibration_comparison.iloc[0]["model"]
extratrees_probability_model = extratrees_probability_models[extratrees_probability_name]
print(f"Soft Voting에 사용할 ExtraTrees: {extratrees_probability_name}")

,model,cv_brier,cv_brier_std,test_brier,test_auc,test_accuracy
0,ExtraTrees_2nd,0.166519,0.018858,0.169055,0.830073,0.790909
1,ExtraTrees_sigmoid_ensemble,0.166879,0.021598,0.168397,0.829410,0.790909
2,ExtraTrees_sigmoid_small,0.167157,0.023154,0.167284,0.830073,0.790909


Soft Voting에 사용할 ExtraTrees: ExtraTrees_2nd


## 10. Soft Voting

각 모델의 Won 확률을 동일한 가중치로 평균합니다. 먼저 확률 성능 상위 두 구조인
LogisticRegression과 TabICL을 평균하고, 다음 후보에서 ExtraTrees와 CatBoost를 추가합니다.
가중치를 같은 Train CV에서 반복 탐색하면 작은 데이터의 CV에 과적합할 수 있어 2차에서는
균등 가중치만 비교합니다.

In [13]:
model_voting_two = VotingClassifier(
    estimators=[
        ("logistic", clone(search_logistic_phase2.best_estimator_)),
        ("tabicl", clone(search_tabicl_phase2.best_estimator_)),
    ],
    voting="soft",
    n_jobs=1,
)
cv_scores_voting_two = cross_val_score(
    model_voting_two,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
)
model_voting_two.fit(X_train, y_train)
test_metrics_voting_two = calculate_test_metrics(model_voting_two)

print(f"LR + TabICL CV Brier: {-cv_scores_voting_two.mean():.6f}")
print(f"LR + TabICL Test: {test_metrics_voting_two}")

LR + TabICL CV Brier: 0.160846
LR + TabICL Test: {'brier': 0.1659468622623057, 'auc': 0.830736074270557, 'accuracy': 0.7909090909090909}


In [14]:
model_voting_three = VotingClassifier(
    estimators=[
        ("logistic", clone(search_logistic_phase2.best_estimator_)),
        ("tabicl", clone(search_tabicl_phase2.best_estimator_)),
        ("extra_trees", clone(extratrees_probability_model)),
    ],
    voting="soft",
    n_jobs=1,
)
cv_scores_voting_three = cross_val_score(
    model_voting_three,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
)
model_voting_three.fit(X_train, y_train)
test_metrics_voting_three = calculate_test_metrics(model_voting_three)

print(f"LR + TabICL + ExtraTrees CV Brier: {-cv_scores_voting_three.mean():.6f}")
print(f"LR + TabICL + ExtraTrees Test: {test_metrics_voting_three}")

LR + TabICL + ExtraTrees CV Brier: 0.162007
LR + TabICL + ExtraTrees Test: {'brier': 0.16615048272445646, 'auc': 0.8310676392572944, 'accuracy': 0.8}


In [15]:
model_voting_four = VotingClassifier(
    estimators=[
        ("logistic", clone(search_logistic_phase2.best_estimator_)),
        ("tabicl", clone(search_tabicl_phase2.best_estimator_)),
        ("extra_trees", clone(extratrees_probability_model)),
        ("catboost", clone(search_catboost_phase2.best_estimator_)),
    ],
    voting="soft",
    n_jobs=1,
)
cv_scores_voting_four = cross_val_score(
    model_voting_four,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
)
model_voting_four.fit(X_train, y_train)
test_metrics_voting_four = calculate_test_metrics(model_voting_four)

print(f"LR + TabICL + ExtraTrees + CatBoost CV Brier: {-cv_scores_voting_four.mean():.6f}")
print(f"LR + TabICL + ExtraTrees + CatBoost Test: {test_metrics_voting_four}")

LR + TabICL + ExtraTrees + CatBoost CV Brier: 0.163518
LR + TabICL + ExtraTrees + CatBoost Test: {'brier': 0.1652891486792711, 'auc': 0.832393899204244, 'accuracy': 0.7909090909090909}


## 11. Stacking

개별 튜닝이 끝난 모델을 새 estimator로 복제해 Stacking에 넣습니다. ExtraTrees 보정기를
Stacking 안에 넣으면 Stacking CV와 보정 CV가 중첩되므로 여기서는 보정 전 ExtraTrees를
사용하고, 최종 LogisticRegression이 세 모델의 Won 확률을 조합하게 합니다.

In [16]:
model_stacking = StackingClassifier(
    estimators=[
        ("logistic", clone(search_logistic_phase2.best_estimator_)),
        ("tabicl", clone(search_tabicl_phase2.best_estimator_)),
        ("extra_trees", clone(search_extratrees_phase2.best_estimator_)),
    ],
    final_estimator=LogisticRegressionCV(
        Cs=[0.03, 0.1, 0.3, 1.0],
        cv=5,
        scoring="neg_brier_score",
        max_iter=2000,
        n_jobs=1,
        random_state=RANDOM_STATE,
    ),
    cv=5,
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=1,
)
cv_scores_stacking = cross_val_score(
    model_stacking,
    X_train,
    y_train,
    cv=cv5,
    scoring="neg_brier_score",
    n_jobs=1,
)
model_stacking.fit(X_train, y_train)
test_metrics_stacking = calculate_test_metrics(model_stacking)

print(f"Stacking CV Brier: {-cv_scores_stacking.mean():.6f}")
print(f"Stacking Test: {test_metrics_stacking}")

Stacking CV Brier: 0.163075
Stacking Test: {'brier': 0.1648916510727235, 'auc': 0.8304045092838195, 'accuracy': 0.8090909090909091}


## 12. 2차 최종 비교

모델과 앙상블은 모두 같은 5-Fold CV Brier로 비교하고, 평균 Brier가 가장 낮은 모델을
선택합니다. Test Brier·AUC·Accuracy는 결과 확인용이며 선택 기준에는 사용하지 않습니다.
임계값 조정은 3차 노트북에서 별도로 진행합니다.


In [17]:
phase2_models = {
    **phase2_individual_models,
    "ExtraTrees_sigmoid_small": model_extratrees_sigmoid_small,
    "ExtraTrees_sigmoid_ensemble": model_extratrees_sigmoid_ensemble,
    "SoftVoting_LR_TabICL": model_voting_two,
    "SoftVoting_LR_TabICL_ET": model_voting_three,
    "SoftVoting_LR_TabICL_ET_CatBoost": model_voting_four,
    "Stacking_LR_TabICL_ET": model_stacking,
}
phase2_best_params = {
    **{name: search.best_params_ for name, search in phase2_individual_searches.items()},
    "ExtraTrees_sigmoid_small": {"method": "sigmoid", "ensemble": False},
    "ExtraTrees_sigmoid_ensemble": {"method": "sigmoid", "ensemble": True},
    "SoftVoting_LR_TabICL": {"voting": "soft", "weights": [1, 1]},
    "SoftVoting_LR_TabICL_ET": {"voting": "soft", "weights": [1, 1, 1]},
    "SoftVoting_LR_TabICL_ET_CatBoost": {
        "voting": "soft",
        "weights": [1, 1, 1, 1],
    },
    "Stacking_LR_TabICL_ET": {
        "final_estimator": "LogisticRegressionCV",
        "passthrough": False,
    },
}

individual_names = [
    "LogisticRegression_2nd",
    "MultinomialNB_2nd",
    "ExtraTrees_2nd",
    "CatBoost_2nd",
    "TabICL_2nd",
    "ExtraTrees_sigmoid_small",
    "ExtraTrees_sigmoid_ensemble",
]
phase2_fold_briers = {
    **phase2_individual_fold_briers,
    "ExtraTrees_sigmoid_small": -cv_scores_extratrees_sigmoid_small,
    "ExtraTrees_sigmoid_ensemble": -cv_scores_extratrees_sigmoid_ensemble,
    "SoftVoting_LR_TabICL": -cv_scores_voting_two,
    "SoftVoting_LR_TabICL_ET": -cv_scores_voting_three,
    "SoftVoting_LR_TabICL_ET_CatBoost": -cv_scores_voting_four,
    "Stacking_LR_TabICL_ET": -cv_scores_stacking,
}
phase2_test_metrics = {
    **phase2_individual_test_metrics,
    "ExtraTrees_sigmoid_small": extratrees_probability_test_metrics["ExtraTrees_sigmoid_small"],
    "ExtraTrees_sigmoid_ensemble": extratrees_probability_test_metrics[
        "ExtraTrees_sigmoid_ensemble"
    ],
    "SoftVoting_LR_TabICL": test_metrics_voting_two,
    "SoftVoting_LR_TabICL_ET": test_metrics_voting_three,
    "SoftVoting_LR_TabICL_ET_CatBoost": test_metrics_voting_four,
    "Stacking_LR_TabICL_ET": test_metrics_stacking,
}

phase2_comparison = pd.DataFrame(
    [
        {
            "model": name,
            "type": "individual" if name in individual_names else "ensemble",
            "cv_brier": float(phase2_fold_briers[name].mean()),
            "cv_brier_std": float(phase2_fold_briers[name].std(ddof=1)),
            **{f"test_{metric}": value for metric, value in phase2_test_metrics[name].items()},
            "best_params": phase2_best_params[name],
        }
        for name in phase2_models
    ]
).sort_values("cv_brier", ignore_index=True)

selected_model_name = phase2_comparison.iloc[0]["model"]
selected_model = phase2_models[selected_model_name]

display(phase2_comparison)
print(f"Brier 기준 선택 모델: {selected_model_name}")
print(f"선택 모델 CV Brier: {phase2_fold_briers[selected_model_name].mean():.6f}")

,model,type,cv_brier,cv_brier_std,test_brier,test_auc,test_accuracy,best_params
0,SoftVoting_LR_TabICL,ensemble,0.160846,0.017887,0.165947,0.830736,0.790909,"{'voting': 'soft', 'weights': [1, 1]}"
1,LogisticRegression_2nd,individual,0.161561,0.016313,0.169427,0.818800,0.754545,{'classifier__C': 0.1}
2,SoftVoting_LR_TabICL_ET,ensemble,0.162007,0.018284,0.166150,0.831068,0.800000,"{'voting': 'soft', 'weights': [1, 1, 1]}"
3,MultinomialNB_2nd,individual,0.162234,0.015349,0.177860,0.825763,0.763636,"{'ensemble': False, 'estimator__classifier__al..."
4,TabICL_2nd,individual,0.162569,0.019479,0.164790,0.837367,0.790909,"{'average_logits': True, 'feat_shuffle_method'..."
5,Stacking_LR_TabICL_ET,ensemble,0.163075,0.019370,0.164892,0.830405,0.809091,"{'final_estimator': 'LogisticRegressionCV', 'p..."
6,SoftVoting_LR_TabICL_ET_CatBoost,ensemble,0.163518,0.019062,0.165289,0.832394,0.790909,"{'voting': 'soft', 'weights': [1, 1, 1, 1]}"
7,ExtraTrees_2nd,individual,0.166519,0.018858,0.169055,0.830073,0.790909,"{'classifier__criterion': 'gini', 'classifier_..."
8,ExtraTrees_sigmoid_ensemble,individual,0.166879,0.021598,0.168397,0.829410,0.790909,"{'method': 'sigmoid', 'ensemble': True}"
9,ExtraTrees_sigmoid_small,individual,0.167157,0.023154,0.167284,0.830073,0.790909,"{'method': 'sigmoid', 'ensemble': False}"


Brier 기준 선택 모델: SoftVoting_LR_TabICL
선택 모델 CV Brier: 0.160846


## 13. 3차 입력값 저장

선택된 모델의 Train OOF 확률과 Test 확률을 저장합니다. 3차 노트북은 이 값만 읽어
임계값을 조정하므로 2차 모델을 다시 학습하거나 직렬화하지 않습니다.


In [18]:
selected_oof_probabilities = cross_val_predict(
    clone(selected_model),
    X_train,
    y_train,
    cv=cv5,
    method="predict_proba",
    n_jobs=1,
)[:, 1]
selected_test_probabilities = won_probabilities(selected_model, X_test)
selected_oof_predictions = (selected_oof_probabilities >= 0.5).astype(int)

selected_oof_metrics = pd.DataFrame(
    [
        {
            "model": selected_model_name,
            "oof_brier": brier_score_loss(y_train, selected_oof_probabilities),
            "oof_auc": roc_auc_score(y_train, selected_oof_probabilities),
            "oof_accuracy": accuracy_score(y_train, selected_oof_predictions),
        }
    ]
)
display(selected_oof_metrics)

threshold_input_path = Path("../pipeline/artifacts/deal-model-phase2-predictions.npz").resolve()
threshold_input_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    threshold_input_path,
    schema_version=np.asarray(1, dtype=np.int64),
    source_sha256=np.asarray(SOURCE_SHA256),
    selected_model_name=np.asarray(selected_model_name),
    selected_cv_brier=np.asarray(
        phase2_fold_briers[selected_model_name].mean(),
        dtype=float,
    ),
    random_state=np.asarray(RANDOM_STATE, dtype=np.int64),
    feature_names=np.asarray(FEATURE_NAMES, dtype=str),
    train_index=X_train.index.to_numpy(dtype=np.int64),
    test_index=X_test.index.to_numpy(dtype=np.int64),
    y_train=y_train.to_numpy(dtype=np.int64),
    y_test=y_test.to_numpy(dtype=np.int64),
    oof_won_probability=selected_oof_probabilities.astype(float),
    test_won_probability=selected_test_probabilities.astype(float),
)
print(f"3차 입력값 저장: {threshold_input_path}")

,model,oof_brier,oof_auc,oof_accuracy
0,SoftVoting_LR_TabICL,0.160846,0.841649,0.772549


3차 입력값 저장: backend/pipeline/artifacts/deal-model-phase2-predictions.npz


## 14. 최종 모델 번들 저장

3차에서 운영 임계값을 확정한 뒤 이 셀을 실행합니다. LogisticRegression 파이프라인과
TabICL을 각각 저장하고, Soft Voting 구성·임계값·파일 해시를 메타데이터에 기록합니다.
TabICL은 외부 체크포인트 없이 불러올 수 있도록 모델 가중치와 학습 문맥을 포함합니다.


In [19]:
import json
import platform
from datetime import UTC, datetime
from importlib.metadata import version

import joblib
import sklearn

MODEL_VERSION = "deal-soft-voting-lr-tabicl-v1"
OPERATING_THRESHOLD = 0.50
ARTIFACT_DIR = Path("../pipeline/artifacts").resolve()
LOGISTIC_MODEL_PATH = ARTIFACT_DIR / f"{MODEL_VERSION}-logistic.joblib"
TABICL_MODEL_PATH = ARTIFACT_DIR / f"{MODEL_VERSION}-tabicl.pkl"
MODEL_METADATA_PATH = ARTIFACT_DIR / f"{MODEL_VERSION}.json"

assert selected_model_name == "SoftVoting_LR_TabICL"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 선택과 평가가 끝난 구성을 전체 365건에 다시 학습해 배포용 모델로 만듭니다.
final_model = clone(selected_model)
final_model.fit(X, y)
logistic_component = final_model.named_estimators_["logistic"]
tabicl_component = final_model.named_estimators_["tabicl"]
self_check_index = X.index[:16].to_numpy(dtype=np.int64)
self_check_X = X.loc[self_check_index]
expected_probabilities = won_probabilities(final_model, self_check_X)

joblib.dump(logistic_component, LOGISTIC_MODEL_PATH, compress=3)
tabicl_component.save(
    TABICL_MODEL_PATH,
    save_model_weights=True,
    save_training_data=True,
    save_kv_cache=False,
)

restored_logistic = joblib.load(LOGISTIC_MODEL_PATH)
restored_tabicl = TabICLClassifier.load(TABICL_MODEL_PATH, device=tabicl_device)
restored_probabilities = (
    won_probabilities(restored_logistic, self_check_X)
    + won_probabilities(restored_tabicl, self_check_X)
) / 2
np.testing.assert_allclose(
    restored_probabilities,
    expected_probabilities,
    rtol=1e-5,
    atol=1e-6,
)


def artifact_sha256(path):
    """큰 모델 파일을 메모리에 모두 올리지 않고 SHA-256을 계산합니다."""
    digest = hashlib.sha256()
    with path.open("rb") as artifact_file:
        for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


artifact_files = {
    "logistic": {
        "path": LOGISTIC_MODEL_PATH.name,
        "format": "joblib",
        "sha256": artifact_sha256(LOGISTIC_MODEL_PATH),
        "size_bytes": LOGISTIC_MODEL_PATH.stat().st_size,
    },
    "tabicl": {
        "path": TABICL_MODEL_PATH.name,
        "format": "tabicl-native-with-weights-and-training-context",
        "sha256": artifact_sha256(TABICL_MODEL_PATH),
        "size_bytes": TABICL_MODEL_PATH.stat().st_size,
    },
}
metadata = {
    "schema_version": 1,
    "model_version": MODEL_VERSION,
    "selected_model": selected_model_name,
    "ensemble": {
        "method": "soft_voting",
        "components": ["logistic", "tabicl"],
        "weights": [0.5, 0.5],
    },
    "operating_threshold": OPERATING_THRESHOLD,
    "feature_names": list(FEATURE_NAMES),
    "target": {"Lost": 0, "Won": 1},
    "data": {
        "source_sha256": source_sha256,
        "raw_rows": raw_rows,
        "deduplicated_rows": deduplicated_rows,
        "synthetic_unknowns": False,
    },
    "evaluation_split": {
        "train_size": 0.7,
        "test_size": 0.3,
        "random_state": RANDOM_STATE,
        "stratified": True,
        "training_rows": len(X_train),
        "test_rows": len(X_test),
    },
    "final_fit": {
        "rows": len(X),
        "uses_all_deduplicated_rows": True,
    },
    "metrics": {
        "cv_brier": float(phase2_fold_briers[selected_model_name].mean()),
        "oof_brier": float(selected_oof_metrics.iloc[0]["oof_brier"]),
        "oof_auc": float(selected_oof_metrics.iloc[0]["oof_auc"]),
        "oof_accuracy": float(selected_oof_metrics.iloc[0]["oof_accuracy"]),
        "test_brier": float(phase2_test_metrics[selected_model_name]["brier"]),
        "test_auc": float(phase2_test_metrics[selected_model_name]["auc"]),
        "test_accuracy": float(phase2_test_metrics[selected_model_name]["accuracy"]),
    },
    "files": artifact_files,
    "versions": {
        "python": platform.python_version(),
        "joblib": version("joblib"),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "tabicl": version("tabicl"),
        "torch": torch.__version__,
    },
    "saved_at_utc": datetime.now(UTC).isoformat(),
    "self_check": {
        "row_indices": self_check_index.tolist(),
        "won_probabilities": expected_probabilities.tolist(),
        "rtol": 1e-5,
        "atol": 1e-6,
        "status": "reloaded_ensemble_probabilities_match",
    },
}
MODEL_METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(f"LogisticRegression: {LOGISTIC_MODEL_PATH}")
print(f"TabICL: {TABICL_MODEL_PATH}")
print(f"메타데이터: {MODEL_METADATA_PATH}")
print("재로드 확률 검증: 통과")

LogisticRegression: backend/pipeline/artifacts/deal-soft-voting-lr-tabicl-v1-logistic.joblib
TabICL: backend/pipeline/artifacts/deal-soft-voting-lr-tabicl-v1-tabicl.pkl
메타데이터: backend/pipeline/artifacts/deal-soft-voting-lr-tabicl-v1.json
재로드 확률 검증: 통과
